# AnimeGANv3 — Video to Animation on Colab T4
Open with a GPU runtime (T4), then **Runtime → Run all**. The notebook installs the inference stack, checks CUDA, asks you to upload an ONNX model and a video, renders it, and reports timing.

In [ ]:
!nvidia-smi
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -q numpy Pillow tqdm opencv-python-headless 'onnxruntime-gpu>=1.18'


In [ ]:
import onnxruntime as ort
print('ORT device:', ort.get_device())
print('Providers:', ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDAExecutionProvider unavailable. Select a GPU runtime and restart the session.'


## Upload model + video
Upload one AnimeGANv3 `.onnx` model and one input video.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
names = list(uploaded)
models = [n for n in names if n.lower().endswith('.onnx')]
videos = [n for n in names if Path(n).suffix.lower() in {'.mp4','.mov','.avi','.mkv','.webm'}]
assert models, 'Upload an .onnx AnimeGANv3 model.'
assert videos, 'Upload an input video.'
MODEL = str(Path(models[0]).resolve())
VIDEO = str(Path(videos[0]).resolve())
print('Model:', MODEL)
print('Video:', VIDEO)


In [ ]:
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,avg_frame_rate,nb_frames,duration -of default=noprint_wrappers=1 "$VIDEO"


In [ ]:
import os, subprocess, time
from pathlib import Path

os.makedirs('output', exist_ok=True)
cmd = ['python','tools/video2anime.py','-i',VIDEO,'-o','output','-m',MODEL,'-d','gpu']
print('Running:', ' '.join(cmd))
start = time.perf_counter()
subprocess.run(cmd, check=True)
elapsed = time.perf_counter() - start
print(f'\nWall time: {elapsed:.2f} s')


In [ ]:
import cv2
cap = cv2.VideoCapture(VIDEO)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
duration = frames / fps if fps else 0
cap.release()
render_fps = frames / elapsed if elapsed else 0
rtf = elapsed / duration if duration else 0
print(f'Frames: {frames}')
print(f'Source FPS: {fps:.3f}')
print(f'Source duration: {duration:.2f} s')
print(f'Render throughput: {render_fps:.2f} FPS')
print(f'Realtime factor: {rtf:.2f}x')


In [ ]:
from pathlib import Path
outs = sorted(Path('output').glob('*'), key=lambda x: x.stat().st_mtime, reverse=True)
print('Output files:')
for f in outs[:10]: print(f)
